# Post-Calibration Configuration Guide

After building a machine with default macros, this notebook shows how to
persist calibrated values onto the QUAM state objects. Every update flows
through the same path that `machine.save()` serialises, so the calibration
survives a save/load round-trip.

**Sections:**

1. **Setup** — Build the tutorial machine and inspect defaults.
2. **XY Drive** — Set calibrated amplitude, pulse duration, and drive frequency.
3. **Voltage Points** — Overwrite gate voltages with calibrated values.
4. **Initialize & Empty Timing** — Tune ramp and hold durations.
5. **PSB Measurement Chain** — Configure the full `Measure1Q` → `MeasurePSBPair` → `SensorDotMeasure` pipeline.
6. **Save & Load** — Verify calibrated values persist through serialisation.

**Runs without QM hardware** — no `qm.open()` or `machine.connect()` required.

## 1. Setup

Build the tutorial machine using the combined wiring workflow. The machine
arrives with default macros and pulses already wired by `build_quam`.

In [1]:
from quam_builder.architecture.quantum_dots.examples.tutorial_machine import build_tutorial_machine
from quam_builder.architecture.quantum_dots.operations.names import VoltagePointName

machine = build_tutorial_machine()

q1 = machine.qubits["q1"]
q2 = machine.qubits["q2"]
pair = machine.quantum_dot_pairs["virtual_dot_1_virtual_dot_2_pair"]
sd = machine.sensor_dots["virtual_sensor_1"]

print("Qubit macros:", sorted(str(k) for k in q1.macros.keys()))
print("Pair macros:", sorted(str(k) for k in pair.macros.keys()))
print("Sensor macros:", sorted(str(k) for k in sd.macros.keys()))
print()
print("q1.preferred_readout_quantum_dot:", q1.preferred_readout_quantum_dot)
print("q2.preferred_readout_quantum_dot:", q2.preferred_readout_quantum_dot)
print("Pair sensor dots:", pair.sensor_dots)

2026-04-14 15:27:06,993 - qm - INFO     - Starting session: 0a597f4e-0399-492f-a6e1-a998b13249d0
Qubit macros: ['-x90', '-y90', 'I', 'align', 'empty', 'initialize', 'measure', 'wait', 'x', 'x180', 'x90', 'x_neg90', 'xy_drive', 'y', 'y180', 'y90', 'y_neg90', 'z', 'z180', 'z90', 'z_neg90']
Pair macros: ['align', 'empty', 'initialize', 'measure', 'wait']
Sensor macros: ['align', 'measure', 'wait']

q1.preferred_readout_quantum_dot: virtual_dot_2
q2.preferred_readout_quantum_dot: virtual_dot_1
Pair sensor dots: ['#/sensor_dots/virtual_sensor_1']


/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/python3.11/site-packages/quam/core/quam_classes.py:289: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: ReadoutResonatorSingle
  warnings.warn(
/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/python3.11/site-packages/quam/core/quam_classes.py:289: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: VirtualGateSet
  warnings.warn(
/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/python3.11/site-packages/quam/core/quam_classes.py:289: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: SensorDot
  warnings.warn(
/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/python3

## 2. XY Drive — Single Source of Truth

All single-qubit gate macros (`x`, `y`, `x/2`, `y/2`, etc.) delegate to a
single `XYDriveMacro` instance at `qubit.macros["xy_drive"]`. This macro
owns the **reference amplitude** and points at a **reference pulse** on the
qubit's XY drive channel.

The `update()` method is the primary API for persisting calibrated values:

| Parameter | Effect |
|-----------|--------|
| `amplitude` | Set `reference_amplitude` (absolute π-rotation voltage) |
| `amplitude_scale` | Multiply current `reference_amplitude` by a factor |
| `duration` | Set the reference pulse `length` in ns (sigma auto-scales via `sigma_ratio`) |
| `frequency` | Set `qubit.xy.intermediate_frequency` (absolute Hz) |
| `frequency_offset` | Add an offset to the current IF |

Sub-π rotations (X/2, Y/2, etc.) are derived automatically by scaling
amplitude proportionally to `angle / reference_angle`.

In [2]:
xy = q1.macros["xy_drive"]
pulse = xy.reference_pulse

print("Before calibration:")
print(f"  reference_amplitude: {xy.reference_amplitude}")
print(f"  reference_angle:     {xy.reference_angle:.4f} rad (π)")
print(f"  pulse name:          {xy.reference_pulse_name}")
print(f"  pulse length:        {pulse.length} ns")
print(f"  pulse sigma_ratio:   {pulse.sigma_ratio}")
print(f"  pulse sigma:         {pulse.sigma:.1f} ns")
print(f"  IF frequency:        {q1.xy.intermediate_frequency} Hz")

xy.update(
    amplitude=0.35,
    duration=40,
    frequency=50e6,
)

print("\nAfter calibration:")
print(f"  reference_amplitude: {xy.reference_amplitude}")
print(f"  pulse length:        {pulse.length} ns")
print(f"  pulse sigma:         {pulse.sigma:.1f} ns  (auto-scaled via sigma_ratio)")
print(f"  IF frequency:        {q1.xy.intermediate_frequency} Hz")

Before calibration:
  reference_amplitude: 1.0
  reference_angle:     3.1416 rad (π)
  pulse name:          gaussian
  pulse length:        1000 ns
  pulse sigma_ratio:   0.16666666666666666
  pulse sigma:         166.7 ns
  IF frequency:        500000000.0 Hz

After calibration:
  reference_amplitude: 0.35
  pulse length:        40 ns
  pulse sigma:         6.7 ns  (auto-scaled via sigma_ratio)
  IF frequency:        50000000.0 Hz


## 3. Voltage Points — Calibrated Gate Voltages

Voltage points define named positions in gate-voltage space (e.g.
`initialize`, `measure`, `empty`). The tutorial machine ships with
placeholder voltages. After tuning, overwrite them with calibrated values
using `add_point()`.

The default `replace_existing_point=True` overwrites any existing point
with the same name.

In [3]:
dot_id = q1.quantum_dot.id

q1.add_point(
    VoltagePointName.INITIALIZE,
    voltages={dot_id: 0.12},
    duration=300,
    replace_existing_point=True,
)
q1.add_point(
    VoltagePointName.MEASURE,
    voltages={dot_id: 0.18},
    duration=500,
    replace_existing_point=True,
)

pair_dot_ids = [qd.id for qd in pair.quantum_dots]
pair.add_point(
    VoltagePointName.INITIALIZE,
    voltages={pair_dot_ids[0]: 0.12, pair_dot_ids[1]: 0.08},
    duration=300,
    replace_existing_point=True,
)
pair.add_point(
    VoltagePointName.MEASURE,
    voltages={pair_dot_ids[0]: 0.18, pair_dot_ids[1]: 0.05},
    duration=500,
    replace_existing_point=True,
)

print(f"q1 initialize: {dot_id} → 0.12 V, 300 ns")
print(f"q1 measure:    {dot_id} → 0.18 V, 500 ns")
print(f"pair initialize: {pair_dot_ids} → [0.12, 0.08] V, 300 ns")
print(f"pair measure:    {pair_dot_ids} → [0.18, 0.05] V, 500 ns")

q1 initialize: virtual_dot_1 → 0.12 V, 300 ns
q1 measure:    virtual_dot_1 → 0.18 V, 500 ns
pair initialize: ['virtual_dot_1', 'virtual_dot_2'] → [0.12, 0.08] V, 300 ns
pair measure:    ['virtual_dot_1', 'virtual_dot_2'] → [0.18, 0.05] V, 500 ns


## 4. Initialize & Empty — Timing Parameters

`InitializeStateMacro` ramps to the initialize voltage point over
`ramp_duration` nanoseconds, then optionally holds for `hold_duration`.
`EmptyStateMacro` steps to the empty voltage point with an optional hold.

Set these directly on the macro instances.

In [4]:
init_macro = q1.macros["initialize"]
empty_macro = q1.macros["empty"]

print("Before:")
print(f"  initialize.ramp_duration: {init_macro.ramp_duration} ns")
print(f"  initialize.hold_duration: {init_macro.hold_duration}")
print(f"  empty.hold_duration:      {empty_macro.hold_duration}")

init_macro.ramp_duration = 48
init_macro.hold_duration = 200
empty_macro.hold_duration = 100

print("\nAfter:")
print(f"  initialize.ramp_duration: {init_macro.ramp_duration} ns")
print(f"  initialize.hold_duration: {init_macro.hold_duration} ns")
print(f"  empty.hold_duration:      {empty_macro.hold_duration} ns")

pair_init = pair.macros["initialize"]
pair_init.ramp_duration = 48
pair_init.hold_duration = 200
print(f"\n  pair.initialize.ramp_duration: {pair_init.ramp_duration} ns")
print(f"  pair.initialize.hold_duration: {pair_init.hold_duration} ns")

Before:
  initialize.ramp_duration: 16 ns
  initialize.hold_duration: None
  empty.hold_duration:      None

After:
  initialize.ramp_duration: 48 ns
  initialize.hold_duration: 200 ns
  empty.hold_duration:      100 ns

  pair.initialize.ramp_duration: 48 ns
  pair.initialize.hold_duration: 200 ns


## 5. PSB Measurement — Full Chain

Single-qubit measurement follows a three-layer delegation:

    qubit.measure()
      → Measure1QMacro
        → finds QuantumDotPair via preferred_readout_quantum_dot
          → MeasurePSBPairMacro
            → steps to "measure" voltage point on the pair
              → SensorDotMeasureMacro
                → readout_resonator.measure("readout")
                → projects: x = I·wI + Q·wQ + offset
                → returns: x > threshold

Each layer has calibration parameters:

| Layer | What to configure |
|-------|-------------------|
| **Qubit** | `preferred_readout_quantum_dot` — which neighbor forms the PSB pair |
| **Pair macro** | `hold_duration` — time held at the measure voltage point |
| **Pair voltages** | Measure voltage point via `add_point` |
| **Sensor** | `readout_thresholds[pair_id]` and `readout_projectors[pair_id]` |

### 5a. Verify Topology

The builder sets `preferred_readout_quantum_dot` automatically from qubit-pair
topology. Verify the mapping is correct for your device.

In [5]:
for name, qubit in machine.qubits.items():
    dot = qubit.quantum_dot.id
    readout_dot = qubit.preferred_readout_quantum_dot
    pair_name = machine.find_quantum_dot_pair(dot, readout_dot)
    print(f"{name}: dot={dot}, readout_dot={readout_dot}, pair={pair_name}")

print(f"\nPair '{pair.id}' sensor dots: {pair.sensor_dots}")

q1: dot=virtual_dot_1, readout_dot=virtual_dot_2, pair=virtual_dot_1_virtual_dot_2_pair
q2: dot=virtual_dot_2, readout_dot=virtual_dot_1, pair=virtual_dot_1_virtual_dot_2_pair

Pair 'virtual_dot_1_virtual_dot_2_pair' sensor dots: ['#/sensor_dots/virtual_sensor_1']


### 5b. Pair Measure Macro

`MeasurePSBPairMacro` steps to the measure voltage point (already updated in
Section 3) and then delegates to the sensor dot. Configure `hold_duration` to
control how long the pair is held at the PSB readout point before measurement.

In [6]:
pair_measure = pair.macros["measure"]
print(f"point:         {pair_measure.point}")
print(f"hold_duration: {pair_measure.hold_duration}")

pair_measure.hold_duration = 500

print(f"\nAfter calibration:")
print(f"  hold_duration: {pair_measure.hold_duration} ns")

point:         measure
hold_duration: None

After calibration:
  hold_duration: 500 ns


### 5c. Sensor Dot — Readout Discrimination

`SensorDotMeasureMacro` measures the readout resonator, then applies a
linear projector in the IQ plane for state discrimination:

    x = I · wI + Q · wQ + offset
    state = x > threshold

Each quantum dot pair has its own threshold and projector stored on the
sensor dot, keyed by pair ID. Set these after running a readout
calibration experiment (e.g. single-shot IQ histograms).

In [7]:
pair_id = pair.id

sd.readout_thresholds[pair_id] = 0.003
sd.readout_projectors[pair_id] = {
    "wI": 0.95,
    "wQ": -0.12,
    "offset": 0.001,
}

print(f"Sensor: {sd.id}")
print(f"  threshold[{pair_id}]:  {sd.readout_thresholds[pair_id]}")
print(f"  projector[{pair_id}]:  {sd.readout_projectors[pair_id]}")

sensor_measure = sd.macros["measure"]
print(f"  pulse_name: {sensor_measure.pulse_name}")

Sensor: virtual_sensor_1
  threshold[virtual_dot_1_virtual_dot_2_pair]:  0.003
  projector[virtual_dot_1_virtual_dot_2_pair]:  {'wI': 0.95, 'wQ': -0.12, 'offset': 0.001}
  pulse_name: readout


### 5d. End-to-End — Build a QUA Program

With all calibration values in place, the full chain compiles into a
valid QUA program. `q1.measure()` triggers the complete
`Measure1QMacro` → `MeasurePSBPairMacro` → `SensorDotMeasureMacro`
pipeline.

In [8]:
from qm import qua

with qua.program() as prog:
    q1.initialize()
    q1.x()
    q1.measure()

print("QUA program compiled successfully with calibrated values.")

QUA program compiled successfully with calibrated values.


## 6. Save & Load Round-Trip

All calibrated values live on QuAM dataclass fields and persist
through `machine.save()` / `machine.load()`.

In [9]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "calibrated_machine.json"
    machine.save(path)
    loaded = type(machine).load(path)

loaded_q1 = loaded.qubits["q1"]
loaded_xy = loaded_q1.macros["xy_drive"]
loaded_pair = loaded.quantum_dot_pairs["virtual_dot_1_virtual_dot_2_pair"]
loaded_sd = loaded.sensor_dots["virtual_sensor_1"]
loaded_pair_id = loaded_pair.id

assert loaded_xy.reference_amplitude == 0.35
assert loaded_xy.reference_pulse.length == 40
assert loaded_q1.xy.intermediate_frequency == 50e6
assert loaded_q1.macros["initialize"].ramp_duration == 48
assert loaded_pair.macros["measure"].hold_duration == 500
assert loaded_sd.readout_thresholds[loaded_pair_id] == 0.003
assert loaded_sd.readout_projectors[loaded_pair_id]["wI"] == 0.95

print("All calibrated values survived save/load:")
print(f"  xy.reference_amplitude:   {loaded_xy.reference_amplitude}")
print(f"  xy pulse length:          {loaded_xy.reference_pulse.length} ns")
print(f"  xy IF frequency:          {loaded_q1.xy.intermediate_frequency} Hz")
print(f"  initialize.ramp_duration: {loaded_q1.macros['initialize'].ramp_duration} ns")
print(f"  pair measure hold:        {loaded_pair.macros['measure'].hold_duration} ns")
print(f"  sensor threshold:         {loaded_sd.readout_thresholds[loaded_pair_id]}")
print(f"  sensor projector:         {loaded_sd.readout_projectors[loaded_pair_id]}")

/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/python3.11/site-packages/quam/core/quam_classes.py:289: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: ReadoutResonatorSingle
  warnings.warn(
/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/python3.11/site-packages/quam/core/quam_classes.py:289: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: VirtualGateSet
  warnings.warn(
/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/python3.11/site-packages/quam/core/quam_classes.py:289: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: QuantumDotPair
  warnings.warn(
/Users/sebastian/Documents/GitHub/quam-builder/.venv/lib/py

All calibrated values survived save/load:
  xy.reference_amplitude:   0.35
  xy pulse length:          40 ns
  xy IF frequency:          50000000.0 Hz
  initialize.ramp_duration: 48 ns
  pair measure hold:        500 ns
  sensor threshold:         0.003
  sensor projector:         {'wI': 0.95, 'wQ': -0.12, 'offset': 0.001}
